Experiment 5 -- Multi-metric fairness monitoring and the impossibility trade-off

In [1]:
# 05_multimetric_monitoring.ipynb
#
# Experiment 5 -- Multi-metric fairness monitoring and the impossibility trade-off
#
# Purpose:
#   Show that optimizing one fairness metric (DIR via group-wise thresholds)
#   does not jointly satisfy equalized odds, empirically illustrating the
#   impossibility result and motivating the multi-metric monitoring design
#   requirement. Compares baseline vs gov_engine on DIR, dTPR, dFPR (30 seeds).
#
# Outputs:
#   results/tables/exp5_multimetric.csv
#   results/figures/exp5_multimetric.(png|pdf)

# ==== Imports and grayscale academic style (600 dpi, PNG+PDF, no captions) ====
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({
    "figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85",
})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def disparate_impact_ratio(y_pred, group):
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean(); r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0) if max(r1, r0) > 0 else np.nan

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(ct, mask):
        idx = (y_true == ct) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    return abs(rate(1, a) - rate(1, b)), abs(rate(0, a) - rate(0, b))



from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

FEATS = ["residential_region", "spending_pattern", "income", "debt_ratio", "pay_history"]

def synth(seed):
    rng = np.random.default_rng(seed); N = 8000
    protected = rng.binomial(1, 0.35, N)
    rr = 1.6 * protected + rng.normal(0, 0.6, N)
    sp = 1.4 * protected + rng.normal(0, 0.6, N)
    inc = rng.normal(0, 1, N); dr = rng.normal(0, 1, N); ph = rng.normal(0, 1, N)
    logit = -0.9*inc + 0.8*dr - 0.7*ph + 1.3*rr + 1.1*sp + rng.normal(0, 0.5, N)
    default = (logit > np.quantile(logit, 0.7)).astype(int)
    X = pd.DataFrame(dict(residential_region=rr, spending_pattern=sp,
                          income=inc, debt_ratio=dr, pay_history=ph))
    return X, default, protected

def run(seed):
    X, y, g = synth(seed)
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, g, test_size=0.3, random_state=seed)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    p = m.predict_proba(Xte.values)[:, 1]
    gg = gte
    yb = (p >= 0.5).astype(int)
    base = (yb == 0).mean()
    t1 = np.quantile(p[gg == 1], base); t0 = np.quantile(p[gg == 0], base)
    yg = np.where(gg == 1, (p >= t1), (p >= t0)).astype(int)
    b_dir = disparate_impact_ratio(yb, gg); b_tpr, b_fpr = equalized_odds_gaps(yte, yb, gg)
    g_dir = disparate_impact_ratio(yg, gg); g_tpr, g_fpr = equalized_odds_gaps(yte, yg, gg)
    return [b_dir, b_tpr, b_fpr, g_dir, g_tpr, g_fpr]

SEEDS = list(range(30))
arr = np.array([run(s) for s in SEEDS])
cols = ["baseline_DIR", "baseline_dTPR", "baseline_dFPR",
        "gov_DIR", "gov_dTPR", "gov_dFPR"]
mean = arr.mean(0); ci = 1.96 * arr.std(0) / np.sqrt(len(SEEDS))
res = pd.DataFrame({"metric": cols, "mean": mean, "ci95": ci})
res.to_csv(os.path.join(TAB_DIR, "exp5_multimetric.csv"), index=False)
print(res.round(3).to_string(index=False))

labels = ["DIR\n(higher=fairer)", "dTPR\n(lower=fairer)", "dFPR\n(lower=fairer)"]
base_vals = mean[[0, 1, 2]]; gov_vals = mean[[3, 4, 5]]
base_ci = ci[[0, 1, 2]]; gov_ci = ci[[3, 4, 5]]
x = np.arange(3); w = 0.38
fig, ax = plt.subplots(figsize=(6.4, 4.3))
ax.bar(x - w/2, base_vals, w, yerr=base_ci, capsize=3, label="Baseline",
       color="#999999", edgecolor="black", linewidth=1.0)
ax.bar(x + w/2, gov_vals, w, yerr=gov_ci, capsize=3, label="AI-Gov-Alt (gov_engine)",
       color="#000000", edgecolor="black", linewidth=1.0)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Metric value"); ax.legend(frameon=True, edgecolor="black")
fig.tight_layout(); save_fig(fig, "exp5_multimetric"); plt.close(fig)
print("Saved multi-metric table and figure.")


       metric  mean  ci95
 baseline_DIR 0.299 0.006
baseline_dTPR 0.210 0.017
baseline_dFPR 0.133 0.008
      gov_DIR 1.000 0.000
     gov_dTPR 0.586 0.003
     gov_dFPR 0.239 0.003


Saved multi-metric table and figure.
